In [1]:
import sys
sys.path.append("..")
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from src.data import load_cuad, TARGET_CLAUSES
from src.split import split_contracts, company_key
from src.abstain import pick_threshold

contracts = load_cuad()
dev_idx, test_idx = split_contracts(contracts)

feat = pd.read_csv("../eval/abstention_features_dev.csv")
feat["i"] = np.tile(dev_idx, len(TARGET_CLAUSES))
feat["company"] = [company_key(contracts[i]["title"]) for i in feat["i"]]

assert feat["clause"].tolist() == [cl for cl in TARGET_CLAUSES for _ in dev_idx]
assert feat["has_clause"].tolist() == [len(contracts[i]["gold"][cl]) > 0 for cl in TARGET_CLAUSES for i in dev_idx]
print(len(feat), "rows OK")

2499 rows OK


In [2]:
SIGNALS = ["dense_q", "dense_kw", "bm25"]


def cv_eval(d, signal, target_recall, n_splits=5):
    hit_pos = hit_neg = n_pos = n_neg = 0
    for tr, te in GroupKFold(n_splits=n_splits).split(d, groups=d["company"]):
        train, test = d.iloc[tr], d.iloc[te]
        t = pick_threshold(train.loc[train.has_clause, signal], target_recall)
        pos = test.loc[test.has_clause, signal]
        neg = test.loc[~test.has_clause, signal]
        hit_pos += (pos >= t).sum()
        n_pos += len(pos)
        hit_neg += (neg >= t).sum()
        n_neg += len(neg)
    return hit_pos / n_pos, hit_neg / n_neg


rows = []
for target in [0.90, 0.95]:
    for cl in TARGET_CLAUSES:
        d = feat[feat.clause == cl].reset_index(drop=True)
        for s in SIGNALS:
            rec, fa = cv_eval(d, s, target)
            rows.append({
                "target": target, "clause": cl, "signal": s,
                "auc": round(roc_auc_score(d.has_clause, d[s]), 3),
                "found_when_present": round(rec, 2),
                "false_alarm": round(fa, 2),
            })
cv = pd.DataFrame(rows)
cv.to_csv("../eval/abstention_cv_dev.csv", index=False)

for target in [0.90, 0.95]:
    print("target recall", target, "(false alarm rate, lower is better)")
    print(cv[cv.target == target].pivot(index="clause", columns="signal", values="false_alarm"))
    print()

target recall 0.9 (false alarm rate, lower is better)
signal                              bm25  dense_kw  dense_q
clause                                                     
Cap On Liability                    0.24      0.36     0.34
Governing Law                       0.06      0.12     0.24
Non-Compete                         0.45      0.61     0.45
Notice Period To Terminate Renewal  0.61      0.56     0.55
Renewal Term                        0.45      0.51     0.59
Termination For Convenience         0.65      0.59     0.69
Uncapped Liability                  0.50      0.36     0.42

target recall 0.95 (false alarm rate, lower is better)
signal                              bm25  dense_kw  dense_q
clause                                                     
Cap On Liability                    0.33      0.49     0.46
Governing Law                       0.10      0.25     0.31
Non-Compete                         0.54      0.68     0.52
Notice Period To Terminate Renewal  0.64      0.64

In [3]:
best = cv[cv.target == 0.90].sort_values("false_alarm").groupby("clause").first()
best[["signal", "auc", "found_when_present", "false_alarm"]]

,signal,auc,found_when_present,false_alarm
clause,,,,
Cap On Liability,bm25,0.932,0.90,0.24
Governing Law,bm25,0.985,0.90,0.06
Non-Compete,dense_q,0.801,0.88,0.45
Notice Period To Terminate Renewal,dense_q,0.776,0.90,0.55
Renewal Term,bm25,0.879,0.90,0.45
Termination For Convenience,dense_kw,0.682,0.90,0.59
Uncapped Liability,dense_kw,0.855,0.90,0.36


In [4]:
thresholds = {}
for cl in TARGET_CLAUSES:
    d = feat[feat.clause == cl]
    s = best.loc[cl, "signal"]
    thresholds[cl] = {
        "signal": s,
        "threshold": round(pick_threshold(d.loc[d.has_clause, s], 0.90), 4),
        "target_recall": 0.90,
    }

with open("../eval/thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=2)
thresholds

{'Governing Law': {'signal': 'bm25',
  'threshold': 8.7787,
  'target_recall': 0.9},
 'Renewal Term': {'signal': 'bm25', 'threshold': 6.6725, 'target_recall': 0.9},
 'Notice Period To Terminate Renewal': {'signal': 'dense_q',
  'threshold': 0.6661,
  'target_recall': 0.9},
 'Termination For Convenience': {'signal': 'dense_kw',
  'threshold': 0.698,
  'target_recall': 0.9},
 'Non-Compete': {'signal': 'dense_q',
  'threshold': 0.6364,
  'target_recall': 0.9},
 'Cap On Liability': {'signal': 'bm25',
  'threshold': 10.0119,
  'target_recall': 0.9},
 'Uncapped Liability': {'signal': 'dense_kw',
  'threshold': 0.6831,
  'target_recall': 0.9}}